# Routing101 ASR — 3-mode ASR-segment search, RRF, mapped back to keyframes

**Purpose:** search the ASR-transcript layer (not keyframe images) three ways, fuse the
three ranked lists with RRF, then resolve each hit to its nearest real keyframe for display.

| Mode | Source | Space | Text encoder |
|---|---|---|---|
| 2.1 `clip_asr` | `AICDataExtracted/clip_asr/*_asr_clip512.npy` (built into a **new** FAISS index here) | CLIP 512-d | Multilingual-CLIP (`pipeline/clip_encoder.py`) |
| 2.2 `siglip_asr` | `AICDataExtracted/asr_embed/*_asr_siglip768.npy` (built into a FAISS index here) | SigLIP2 768-d | `google/siglip2-base-patch16-384` text tower |
| 2.3 `fuzzy` | `AICDataExtracted/transcripts/*.csv` bulk-indexed into Elasticsearch | lexical | ES `match` query, `fuzziness: AUTO` |
| 2.4 `rrf` | fuses 2.1 + 2.2 + 2.3 | — | — |

Join keys (verified on disk): `transcripts.segment_id == clip_asr_segments.segment_id ==
asr_embed_frames.segment_id`; `asr_embed_frames.frame_id == map-keyframes.n` (already a
direct keyframe pointer, no nearest-timestamp lookup needed for that leg). `clip_asr` and
`fuzzy` hits are segment-level only, so they're mapped to the keyframe whose
`map-keyframes.pts_time` is closest to the segment's `start_sec`.

**Known data gap:** `asr_embed` (SigLIP-ASR) has zero files for lesson `L25` — mode 2.2
simply won't have that lesson's videos in its index; the other two modes + RRF still work.


## 1. Install / imports
Run once. `pip install --break-system-packages faiss-cpu numpy pandas torch transformers sentencepiece pillow elasticsearch` -- drop the flag if you're in a normal venv/conda env.

In [3]:
# !pip install --break-system-packages faiss-cpu numpy pandas torch transformers sentencepiece pillow elasticsearch

import sys
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import faiss
import torch
from IPython.display import display

print("faiss:", faiss.__version__ if hasattr(faiss, "__version__") else "unknown")
print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())

# Reuse pipeline/*.py's text encoders (no fusion/store logic in them) --
# same sys.path trick routing101.py uses so their bare `import config` resolves.
REPO_ROOT = Path.cwd()
PIPELINE_DIR = REPO_ROOT / "pipeline"
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

import config as pconfig  # noqa: E402  (pipeline/config.py)
import clip_encoder        # noqa: E402  (pipeline/clip_encoder.py) -- Multilingual-CLIP text tower


faiss: 1.15.0
torch: 2.13.0+cpu | cuda available: False


## 2. CONFIG — edit this cell

Set `QUERY` and `TOP_K`. Paths point at data outside the repo, same convention as
`pipeline/config.py`. FAISS indices built here are cached on disk under
`index/routing101_asr/` (built once, reused after) -- delete that folder to force a rebuild.

In [4]:
TOP_K = 100
SHOW_TOP_N = 15

# --- source data ---
TRANSCRIPTS_DIR = Path("D:/University/Summ26/AICDataExtracted/transcripts")
CLIP_ASR_DIR = Path("D:/University/Summ26/AICDataExtracted/clip_asr")
ASR_EMBED_DIR = Path("D:/University/Summ26/AICDataExtracted/asr_embed")
MAP_KEYFRAMES_DIR = Path("D:/University/Summ26/AICData/map-keyframes")
THUMBNAIL_ROOT = Path("D:/University/Summ26/AICData/keyframes")

# --- on-disk FAISS cache for this notebook ---
INDEX_DIR = REPO_ROOT / "index" / "routing101_asr"
INDEX_DIR.mkdir(parents=True, exist_ok=True)
CLIP_ASR_FAISS = INDEX_DIR / "clip_asr_flat_ip.index"
CLIP_ASR_META = INDEX_DIR / "meta_clip_asr.csv"
SIGLIP_ASR_FAISS = INDEX_DIR / "siglip_asr_flat_ip.index"
SIGLIP_ASR_META = INDEX_DIR / "meta_siglip_asr.csv"

# --- SigLIP2 text tower (query side only) ---
SIGLIP2_MODEL_ID = "google/siglip2-base-patch16-384"

# --- Elasticsearch (simple local instance, no auth/SSL) ---
# docker run -d --name es -p 9200:9200 -e "discovery.type=single-node" \
#     -e "xpack.security.enabled=false" docker.elastic.co/elasticsearch/elasticsearch:8.15.0
ES_HOST = "http://localhost:9200"
ES_INDEX = "asr_segments"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


## 3. Shared helpers
`video_id_from_filename` strips a known suffix, `l2_normalize` for cosine-via-inner-product, and a small per-video `map-keyframes` cache used by the keyframe-mapping step.

In [5]:
def video_id_from_filename(npy_path: Path, suffix: str) -> str:
    return npy_path.stem[: -len(suffix)] if npy_path.stem.endswith(suffix) else npy_path.stem


def l2_normalize(mat: np.ndarray) -> np.ndarray:
    mat = mat.astype("float32", copy=True)
    faiss.normalize_L2(mat)
    return mat


_map_keyframes_cache: dict = {}


def load_map_keyframes(video_id: str):
    if video_id not in _map_keyframes_cache:
        path = MAP_KEYFRAMES_DIR / f"{video_id}.csv"
        _map_keyframes_cache[video_id] = pd.read_csv(path) if path.exists() else None
    return _map_keyframes_cache[video_id]


def nearest_keyframe_by_time(video_id: str, t: float):
    '''Nearest map-keyframes row (by pts_time) to timestamp t. Returns
    (n, pts_time, frame_idx) or (None, None, None) if no map-keyframes file.'''
    mk = load_map_keyframes(video_id)
    if mk is None or mk.empty or pd.isna(t):
        return None, None, None
    idx = (mk["pts_time"] - t).abs().idxmin()
    row = mk.loc[idx]
    return int(row["n"]), float(row["pts_time"]), int(row["frame_idx"])


def keyframe_by_n(video_id: str, n: int):
    '''Direct n-indexed lookup (used for the SigLIP-ASR leg, which already
    carries frame_id == map-keyframes.n).'''
    mk = load_map_keyframes(video_id)
    if mk is None or pd.isna(n):
        return None, None
    hit = mk.loc[mk["n"] == int(n)]
    if hit.empty:
        return None, None
    row = hit.iloc[0]
    return float(row["pts_time"]), int(row["frame_idx"])


def thumbnail_path(video_id: str, n) -> str:
    if n is None or pd.isna(n):
        return ""
    return str(THUMBNAIL_ROOT / video_id / f"{int(n):03d}.jpg")


## 4. Step 1 — embed `clip_asr` into a new FAISS index

`clip_asr/*_asr_clip512.npy` vectors are precomputed CLIP-space embeddings of ASR segment
text, one `.npy` (+ `_segments.csv` sidecar) per video, **not unit-normalized on disk**.
Cached to `index/routing101_asr/clip_asr_flat_ip.index` + `meta_clip_asr.csv` so this only
runs once.

In [6]:
def _build_clip_asr_index():
    npy_paths = sorted(CLIP_ASR_DIR.glob("*_asr_clip512.npy"))
    print(f"[clip_asr] building index over {len(npy_paths)} videos")

    index = faiss.IndexFlatIP(512)
    rows = []
    gid = 0
    for npy_path in npy_paths:
        video_id = video_id_from_filename(npy_path, "_asr_clip512")
        seg_path = CLIP_ASR_DIR / f"{video_id}_asr_clip512_segments.csv"
        if not seg_path.exists():
            print(f"  skip {video_id}: missing segments csv")
            continue
        vecs = l2_normalize(np.load(npy_path))
        segs = pd.read_csv(seg_path)
        if len(segs) != vecs.shape[0]:
            print(f"  skip {video_id}: row mismatch ({len(segs)} segments vs {vecs.shape[0]} vectors)")
            continue
        index.add(vecs)
        for _, r in segs.iterrows():
            rows.append((gid, video_id, int(r["segment_id"]), float(r["start_sec"]), float(r["end_sec"]), r["text"]))
            gid += 1

    faiss.write_index(index, str(CLIP_ASR_FAISS))
    pd.DataFrame(rows, columns=["global_id", "video_id", "segment_id", "start_sec", "end_sec", "text"]).to_csv(CLIP_ASR_META, index=False)
    print(f"[clip_asr] done: {gid} segments -> {CLIP_ASR_FAISS}")


if not (CLIP_ASR_FAISS.exists() and CLIP_ASR_META.exists()):
    _build_clip_asr_index()

clip_asr_index = faiss.read_index(str(CLIP_ASR_FAISS))
clip_asr_meta = pd.read_csv(CLIP_ASR_META)
print("clip_asr index:", clip_asr_index.ntotal, "vectors")


clip_asr index: 110810 vectors


## 5. Mode 2.1 — CLIP ASR search
Query text encoded with Multilingual-CLIP (`pipeline/clip_encoder.py`), searched against the index built in Step 1.

In [7]:
def search_clip_asr(query: str, k: int = TOP_K) -> pd.DataFrame:
    qvec = l2_normalize(clip_encoder.encode_text([query]))
    n = min(k, clip_asr_index.ntotal)
    scores, ids = clip_asr_index.search(qvec, n)
    rows = []
    for rank, (gid, score) in enumerate(zip(ids[0], scores[0]), start=1):
        if gid == -1:
            continue
        row = clip_asr_meta.iloc[int(gid)]
        rows.append({"rank": rank, "score": float(score), "video_id": row["video_id"],
                      "segment_id": int(row["segment_id"]), "start_sec": row["start_sec"],
                      "end_sec": row["end_sec"], "text": row["text"]})
    return pd.DataFrame(rows)


## 6. Mode 2.2 setup — embed `asr_embed` (SigLIP) into a FAISS index

`asr_embed/*_asr_siglip768.npy` is SigLIP-space, one row per (segment × overlapping
keyframe) -- so `frame_id` in `_frames.csv` is already a direct `map-keyframes.n` pointer,
no nearest-timestamp lookup needed for this leg. Lesson `L25` has no files here (known gap)
and is simply absent from this index. Cached the same way as Step 1.

In [8]:
def _build_siglip_asr_index():
    npy_paths = sorted(ASR_EMBED_DIR.glob("*_asr_siglip768.npy"))
    print(f"[siglip_asr] building index over {len(npy_paths)} videos")

    index = faiss.IndexFlatIP(768)
    rows = []
    gid = 0
    for npy_path in npy_paths:
        video_id = video_id_from_filename(npy_path, "_asr_siglip768")
        frames_path = ASR_EMBED_DIR / f"{video_id}_asr_siglip768_frames.csv"
        if not frames_path.exists():
            print(f"  skip {video_id}: missing frames csv")
            continue
        vecs = l2_normalize(np.load(npy_path))
        frames = pd.read_csv(frames_path)
        if len(frames) != vecs.shape[0]:
            print(f"  skip {video_id}: row mismatch ({len(frames)} frames vs {vecs.shape[0]} vectors)")
            continue
        index.add(vecs)
        for _, r in frames.iterrows():
            rows.append((gid, video_id, int(r["frame_id"]), int(r["segment_id"]),
                         float(r["start_sec"]), float(r["end_sec"]), r["text"]))
            gid += 1

    faiss.write_index(index, str(SIGLIP_ASR_FAISS))
    pd.DataFrame(rows, columns=["global_id", "video_id", "frame_id", "segment_id", "start_sec", "end_sec", "text"]).to_csv(SIGLIP_ASR_META, index=False)
    print(f"[siglip_asr] done: {gid} rows -> {SIGLIP_ASR_FAISS}")


if not (SIGLIP_ASR_FAISS.exists() and SIGLIP_ASR_META.exists()):
    _build_siglip_asr_index()

siglip_asr_index = faiss.read_index(str(SIGLIP_ASR_FAISS))
siglip_asr_meta = pd.read_csv(SIGLIP_ASR_META)
print("siglip_asr index:", siglip_asr_index.ntotal, "vectors")


siglip_asr index: 109728 vectors


In [9]:
_siglip2_state = {}  # lazy singleton: {"model", "processor"}


def _get_siglip2():
    if not _siglip2_state:
        from transformers import AutoModel, AutoProcessor
        model = AutoModel.from_pretrained(SIGLIP2_MODEL_ID).to(DEVICE).eval()
        processor = AutoProcessor.from_pretrained(SIGLIP2_MODEL_ID)
        _siglip2_state.update(model=model, processor=processor)
    return _siglip2_state["model"], _siglip2_state["processor"]


def encode_text_siglip2(texts: list) -> np.ndarray:
    model, processor = _get_siglip2()
    inputs = processor(text=texts, padding="max_length", truncation=True, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.get_text_features(**inputs)
    feats = out.pooler_output if hasattr(out, "pooler_output") else out
    return feats.float().cpu().numpy().astype("float32")


def search_siglip_asr(query: str, k: int = TOP_K) -> pd.DataFrame:
    qvec = l2_normalize(encode_text_siglip2([query]))
    n = min(k, siglip_asr_index.ntotal)
    scores, ids = siglip_asr_index.search(qvec, n)
    rows = []
    for rank, (gid, score) in enumerate(zip(ids[0], scores[0]), start=1):
        if gid == -1:
            continue
        row = siglip_asr_meta.iloc[int(gid)]
        rows.append({"rank": rank, "score": float(score), "video_id": row["video_id"],
                      "segment_id": int(row["segment_id"]), "frame_id": int(row["frame_id"]),
                      "start_sec": row["start_sec"], "end_sec": row["end_sec"], "text": row["text"]})
    return pd.DataFrame(rows)


## 7. Mode 2.3 — Elasticsearch fuzzy search

Simplest possible setup: one flat index over every `transcripts/*.csv` row, a single
`match` query with `fuzziness: "AUTO"`. Requires a local ES reachable at `ES_HOST` (see
the docker one-liner in the CONFIG cell). Bulk-index uses an explicit `_id` per doc so
re-running this cell is idempotent (no duplicate docs).

In [10]:
from elasticsearch import Elasticsearch, helpers

es = Elasticsearch(ES_HOST)


def build_fuzzy_index():
    if not es.indices.exists(index=ES_INDEX):
        es.indices.create(index=ES_INDEX, mappings={"properties": {
            "video_id": {"type": "keyword"},
            "segment_id": {"type": "integer"},
            "start_sec": {"type": "float"},
            "end_sec": {"type": "float"},
            "text": {"type": "text"},
        }})

    def _docs():
        for csv_path in sorted(TRANSCRIPTS_DIR.glob("*.csv")):
            if csv_path.name == "manifest.csv":
                continue
            df = pd.read_csv(csv_path)
            video_id = csv_path.stem
            for _, r in df.iterrows():
                yield {
                    "_index": ES_INDEX,
                    "_id": f"{video_id}_{int(r['segment_id'])}",
                    "_source": {
                        "video_id": video_id,
                        "segment_id": int(r["segment_id"]),
                        "start_sec": float(r["start_sec"]),
                        "end_sec": float(r["end_sec"]),
                        "text": r["text"],
                    },
                }

    n_ok, errors = helpers.bulk(es, _docs(), stats_only=False, raise_on_error=False)
    print(f"[fuzzy] indexed {n_ok} docs, {len(errors)} errors")


def search_fuzzy(query: str, k: int = TOP_K) -> pd.DataFrame:
    """Returns an empty DataFrame (with a warning) instead of raising when ES
    isn't reachable/indexed -- so a missing local ES only drops this one leg
    rather than breaking the run cell / RRF fusion."""
    try:
        resp = es.search(index=ES_INDEX, size=k, query={
            "match": {"text": {"query": query, "fuzziness": "AUTO"}}
        })
    except Exception as e:
        print(f"[fuzzy] search failed ({e}); returning empty results for this leg.")
        return pd.DataFrame(columns=["rank", "score", "video_id", "segment_id", "start_sec", "end_sec", "text"])

    rows = []
    for rank, hit in enumerate(resp["hits"]["hits"], start=1):
        src = hit["_source"]
        rows.append({"rank": rank, "score": float(hit["_score"]), "video_id": src["video_id"],
                      "segment_id": src["segment_id"], "start_sec": src["start_sec"],
                      "end_sec": src["end_sec"], "text": src["text"]})
    return pd.DataFrame(rows)


# Run once to (re-)populate the ES index; safe to re-run (idempotent _id).
try:
    build_fuzzy_index()
except Exception as e:
    print(f"[fuzzy] could not reach Elasticsearch at {ES_HOST}: {e}\n"
          f"        start it first (see docker one-liner in the CONFIG cell) -- "
          f"the other two modes work fine without it.")


[fuzzy] indexed 110810 docs, 0 errors


## 8. Mode 2.4 — Reciprocal Rank Fusion

Adapted from `routing101_app.py`'s `rrf_fuse` -- unweighted `1/(k+rank)` per leg, keyed by
`(video_id, segment_id)` (segment-level, since that's the common granularity across all
three modes; the SigLIP leg's extra `frame_id` per row is carried along on a best-effort
basis for display).

In [11]:
RRF_K = 60


def rrf_fuse(named_dfs: dict, k: int = RRF_K, top_n: int = TOP_K) -> pd.DataFrame:
    '''named_dfs: {leg_name: DataFrame[rank, video_id, segment_id, ...]}'''
    scores: dict = {}
    extra: dict = {}
    for leg_name, df in named_dfs.items():
        if df is None or df.empty:
            continue
        for _, row in df.iterrows():
            key = (row["video_id"], int(row["segment_id"]))
            scores[key] = scores.get(key, 0.0) + 1.0 / (k + row["rank"])
            e = extra.setdefault(key, {"text": row.get("text"), "start_sec": row.get("start_sec"),
                                        "end_sec": row.get("end_sec"), "frame_id": row.get("frame_id")})
            if pd.isna(e.get("frame_id")) and not pd.isna(row.get("frame_id", np.nan)):
                e["frame_id"] = row["frame_id"]

    rows = [{"video_id": vid, "segment_id": sid, "rrf_score": s, **extra[(vid, sid)]}
            for (vid, sid), s in scores.items()]
    out = pd.DataFrame(rows).sort_values("rrf_score", ascending=False).reset_index(drop=True)
    out["rank"] = np.arange(1, len(out) + 1)
    return out.head(top_n)


## 9. Map top-k results back to the nearest keyframe

- SigLIP-ASR hits already carry `frame_id == map-keyframes.n` -- direct lookup.
- CLIP-ASR / fuzzy / RRF hits are segment-level -- mapped to the keyframe whose
  `pts_time` is closest to the segment's `start_sec`.

In [12]:
def attach_keyframe(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return df
    ns, pts_times, frame_idxs, thumbs = [], [], [], []
    for _, row in df.iterrows():
        fid = row.get("frame_id", np.nan)
        if pd.notna(fid):
            pts_time, frame_idx = keyframe_by_n(row["video_id"], fid)
            n = int(fid)
        else:
            n, pts_time, frame_idx = nearest_keyframe_by_time(row["video_id"], row.get("start_sec"))
        ns.append(n)
        pts_times.append(pts_time)
        frame_idxs.append(frame_idx)
        thumbs.append(thumbnail_path(row["video_id"], n))
    out = df.copy()
    out["mapped_n"] = ns
    out["mapped_pts_time"] = pts_times
    out["mapped_frame_idx"] = frame_idxs
    out["thumbnail_path"] = thumbs
    return out


## 10. Run all 3 modes + RRF, mapped to keyframes

In [16]:
QUERY = "triển lãm nghệ thuật"

results_clip_asr = attach_keyframe(search_clip_asr(QUERY, k=TOP_K))
results_siglip_asr = attach_keyframe(search_siglip_asr(QUERY, k=TOP_K))
results_fuzzy = attach_keyframe(search_fuzzy(QUERY, k=TOP_K))
results_rrf = attach_keyframe(rrf_fuse({
    "clip_asr": results_clip_asr,
    "siglip_asr": results_siglip_asr,
    "fuzzy": results_fuzzy,
}, top_n=TOP_K))

cols_leg = ["rank", "score", "video_id", "segment_id", "text", "start_sec", "mapped_n", "mapped_pts_time"]
cols_rrf = ["rank", "rrf_score", "video_id", "segment_id", "text", "start_sec", "mapped_n", "mapped_pts_time"]


def _show(name: str, df: pd.DataFrame, cols: list):
    print(f"--- {name} ---")
    if df is None or df.empty:
        print("(no results)")
    else:
        display(df[cols].head(SHOW_TOP_N))


print(f"QUERY = {QUERY!r}\n")
_show("2.1 clip_asr", results_clip_asr, cols_leg)
_show("2.2 siglip_asr", results_siglip_asr, cols_leg)
_show("2.3 fuzzy (elasticsearch)", results_fuzzy, cols_leg)
_show("2.4 rrf (fused)", results_rrf, cols_rrf)


QUERY = 'triển lãm nghệ thuật'

--- 2.1 clip_asr ---


,rank,score,video_id,segment_id,text,start_sec,mapped_n,mapped_pts_time
0,1,0.890745,L25_V068,324,To the party,722.16,183,720.00
1,2,0.888188,L25_V068,510,Party,1147.20,285,1146.00
2,3,0.886828,L30_V003,10,And my,46.40,17,47.28
3,4,0.884815,L30_V044,29,Song,97.04,57,97.32
4,5,0.884815,L26_V060,40,Song,226.16,97,226.12
5,6,0.884301,L25_V023,473,For the,1081.44,274,1084.24
6,7,0.884293,L25_V075,309,See,697.44,167,696.00
7,8,0.884293,L25_V068,467,See,1054.96,270,1056.00
8,9,0.884293,L25_V041,584,See,1259.28,306,1257.88
9,10,0.884293,L25_V041,494,See,1077.12,276,1077.88


--- 2.2 siglip_asr ---


,rank,score,video_id,segment_id,text,start_sec,mapped_n,mapped_pts_time
0,1,0.969661,L21_V023,176,Triển lãm sẽ mở cửa đến hết ngày hai mươi thán...,1035.04,239,1038.000
1,2,0.969125,L21_V030,111,Triển lãm sẽ diễn ra đến hết ngày ba mươi mốt ...,855.60,212,856.433
2,3,0.957970,L22_V016,182,Triển lãm sẽ kéo dài đến ngày hai mươi bảy thá...,1003.44,255,1004.040
3,4,0.944059,L21_V014,94,Di ri tổ chức triển lãm nhằm hồi sinh nghề in ...,670.40,181,673.900
4,5,0.940163,L21_V008,199,Đây là nỗ lực nhằm phổ biến các tác phẩm nghệ ...,1105.20,263,1106.400
5,6,0.940114,L22_V026,42,Trưng bày chuyên đề cổ động kỳ quan nơi hội tụ...,229.28,59,232.200
6,7,0.938223,L30_V057,5,Về bộ môn nghệ thuật này trên mạng,26.96,9,28.640
7,8,0.938223,L30_V057,5,Về bộ môn nghệ thuật này trên mạng,26.96,8,27.680
8,9,0.933488,L22_V002,101,Ngoài các màn trình diễn âm nhạc và nghệ thuật...,695.04,194,698.000
9,10,0.933488,L22_V002,101,Ngoài các màn trình diễn âm nhạc và nghệ thuật...,695.04,193,695.760


--- 2.3 fuzzy (elasticsearch) ---


,rank,score,video_id,segment_id,text,start_sec,mapped_n,mapped_pts_time
0,1,15.990807,L22_V002,101,Ngoài các màn trình diễn âm nhạc và nghệ thuật...,695.04,193,695.76
1,2,14.751945,L25_V027,401,Về nghệ thuật,1307.28,321,1308.00
2,3,14.751945,L25_V054,210,Nghệ thuật này,935.84,215,936.00
3,4,14.751945,L26_V250,3,Nghệ thuật diễn,19.44,17,19.28
4,5,14.120217,L30_V008,44,Phát triển cái nghệ thuật này hơn thì vì bản t...,178.56,76,179.72
5,6,14.116888,L25_V001,449,Đó nghệ thuật đó,1396.80,326,1398.00
6,7,13.535112,L25_V016,153,À các tỉnh phát triển nghề cá mạnh như là tỉnh...,525.92,153,525.48
7,8,13.111490,L25_V001,450,Mới là nghệ thuật mà người ta ca ngợi gọi là n...,1398.16,326,1398.00
8,9,12.997800,L21_V025,188,Ở loại hình nghệ thuật này,1103.92,268,1103.12
9,10,12.997800,L21_V027,137,Đặc biệt là trường nghệ thuật,867.36,220,866.40


--- 2.4 rrf (fused) ---


,rank,rrf_score,video_id,segment_id,text,start_sec,mapped_n,mapped_pts_time
0,1,0.045172,L22_V002,101,Ngoài các màn trình diễn âm nhạc và nghệ thuật...,695.04,194,698.000
1,2,0.041259,L30_V057,5,Về bộ môn nghệ thuật này trên mạng,26.96,9,28.640
2,3,0.027973,L21_V014,137,Tại triển lãm các nhà in ham vi cuối cùng còn ...,930.40,238,931.900
3,4,0.027799,L21_V027,137,Đặc biệt là trường nghệ thuật,867.36,221,868.667
4,5,0.025642,L27_V007,39,Quy tụ nhiều anh tài văn nhân tham gia sáng tá...,186.96,142,190.360
5,6,0.024846,L21_V002,19,Bên cạnh đó triển lãm cũng tổ chức các hoạt độ...,128.40,36,131.333
6,7,0.024554,L21_V014,94,Di ri tổ chức triển lãm nhằm hồi sinh nghề in ...,670.40,181,673.900
7,8,0.024481,L30_V086,6,Đối với một tác phẩm nghệ thuật thì,35.36,14,35.400
8,9,0.023932,L22_V015,128,Ý tưởng của cuộc triển lãm là đưa nghệ thuật đ...,699.92,166,701.400
9,10,0.023393,L22_V024,83,Những hiện vật được trưng bày trong sự kiện nà...,503.44,133,507.600
